# Printed-label pre-test diffusion-distance selection

This notebook compares partial-diffusion distances 50, 100 and 250 using only the ten
registered **validation normals** (`N09` and `N10`) plus controlled digital defects with
known masks. It does not load `N11`, `N12`, or any real defect image.

Each candidate threshold is calibrated from the untouched validation normals. The
distance with the highest mean synthetic-defect Dice is selected and frozen for the
real test. Synthetic validation supports parameter selection; it is not reported as
real detection accuracy.

Attach only `printed_label_validation_v1.zip` and `printed_label_latest.pt`. Select a
Kaggle GPU, enable Internet, run all cells, and download the result ZIP.


In [ ]:
from pathlib import Path
import csv, importlib.util, json, random, shutil, subprocess, sys, time, zipfile

import numpy as np
import torch
from PIL import Image, ImageDraw, ImageOps
import matplotlib.pyplot as plt

assert Path('/kaggle/input').is_dir(), 'Run this notebook on Kaggle.'
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator first.'

KAGGLE_INPUT=Path('/kaggle/input')
WORK=Path('/kaggle/working/labelinspect')
WORK.mkdir(parents=True,exist_ok=True)
OUTPUT=WORK/'printed_label_model_selection'
OUTPUT.mkdir(parents=True,exist_ok=True)

def find_dataset(search_root):
    found=[]
    for path in search_root.rglob('printed_label_validation_v1'):
        if path.is_dir() and (path/'validation'/'normal').is_dir(): found.append(path)
    return sorted(set(found))

dataset_candidates=find_dataset(KAGGLE_INPUT)
if not dataset_candidates:
    matching=[]
    for archive_path in KAGGLE_INPUT.rglob('*.zip'):
        try:
            with zipfile.ZipFile(archive_path) as archive:
                names=['/'+item.filename.replace('\\','/').lstrip('/') for item in archive.infolist()]
                if any('/printed_label_validation_v1/validation/normal/' in name for name in names): matching.append(archive_path)
        except zipfile.BadZipFile: pass
    if len(matching)==1:
        extraction_root=WORK/'uploaded_validation';extraction_root.mkdir(parents=True,exist_ok=True)
        resolved=extraction_root.resolve()
        with zipfile.ZipFile(matching[0]) as archive:
            for item in archive.infolist():
                target=(extraction_root/item.filename).resolve()
                if target!=resolved and resolved not in target.parents: raise ValueError(f'Unsafe ZIP member: {item.filename}')
            archive.extractall(extraction_root)
        dataset_candidates=find_dataset(extraction_root)
if len(dataset_candidates)!=1:
    raise FileNotFoundError('Expected one printed_label_validation_v1 dataset: '+repr([str(p) for p in dataset_candidates]))
DATA_ROOT=dataset_candidates[0]

all_pt_files=sorted(KAGGLE_INPUT.rglob('printed_label_latest.pt'))
if not all_pt_files:
    extracted_roots=[]
    for data_pickle in KAGGLE_INPUT.rglob('data.pkl'):
        candidate=data_pickle.parent
        if (candidate/'data').is_dir() and (candidate/'version').is_file(): extracted_roots.append(candidate)
    extracted_roots=sorted(set(extracted_roots))
    if len(extracted_roots)==1:
        archive_root=extracted_roots[0]
        rebuilt=WORK/'rebuilt_printed_label_checkpoint.pt'
        with zipfile.ZipFile(rebuilt,'w',compression=zipfile.ZIP_STORED) as archive:
            for source_file in sorted(archive_root.rglob('*')):
                if source_file.is_file(): archive.write(source_file,f'{archive_root.name}/{source_file.relative_to(archive_root).as_posix()}')
        all_pt_files=[rebuilt]
        print('Rebuilt Kaggle-extracted checkpoint:',rebuilt)
if len(all_pt_files)!=1:
    raise FileNotFoundError('Expected one printed-label checkpoint; found: '+repr([str(p) for p in all_pt_files]))
CHECKPOINT=all_pt_files[0]

SEED=230224
IMAGE_SIZE=224
BATCH_SIZE=4
DISTANCES=[50,100,250]
PIXEL_NORMAL_FPR=0.005
IMAGE_SCORE_QUANTILE=0.995
print('GPU:',torch.cuda.get_device_name(0))
print('Validation dataset:',DATA_ROOT)
print('Checkpoint:',CHECKPOINT)
print('Distances:',DISTANCES)


In [ ]:
missing=[]
for package,module in [('timm','timm'),('einops','einops'),('numba','numba'),('scikit-learn','sklearn')]:
    if importlib.util.find_spec(module) is None: missing.append(package)
if missing:
    subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
print('Dependencies ready.')


In [ ]:
script = WORK / 'author_smoke.py'
script.write_text('"""Run a bounded integration check of the author DTU-Net and Tsimplex code.\n\nThis is not training for anomaly detection, not a reproduced result, and not a\nperformance comparison. Downloads only five source files from a pinned commit.\n"""\nfrom __future__ import annotations\nimport argparse\nimport ast\nimport hashlib\nimport importlib.util\nimport json\nimport random\nimport sys\nimport time\nimport types\nimport urllib.error\nimport urllib.request\nfrom pathlib import Path\n\nCOMMIT = \'dc4a9bd2a2a5b1c31223daab4bdfea3f6a5b2990\'\nBASE_URL = f\'https://raw.githubusercontent.com/MAXNORM8650/Annotsim/{COMMIT}/\'\nFILES = [\'src/models/UModels/UDHVT.py\',\'GaussianDiffusion.py\',\n         \'utils/Simplex/constants.py\',\'utils/Simplex/internals.py\',\'utils/Simplex/noise.py\']\nEXPECTED = {\n \'utils/Simplex/constants.py\':\'52bf6ba3e2c0d386fa420382de380093a8dd61f488765cb812b13e25d0be7294\',\n \'utils/Simplex/internals.py\':\'ef67562885dcfe3356acd97784fe10660bf21238be7bcc608e86053c529fd61a\',\n \'src/models/UModels/UDHVT.py\':\'f7303c4dd228a3f5e1ab98d16fe1db7c7abfecfa97f683e89449128c5a03a4c2\',\n \'GaussianDiffusion.py\':\'cdf7a2143a441d20a3250458c0683c53ac1484ef8f0d0927831e66a34b52ec9a\',\n \'utils/Simplex/noise.py\':\'d114b6898369a0299e48f95fe4165fb3d587dcb8d7e257b58aef02077b6249d3\',\n}\n\n\ndef fetch_sources(root):\n    hashes={}\n    for name in FILES:\n        destination=root/name\n        destination.parent.mkdir(parents=True,exist_ok=True)\n        if not destination.exists():\n            print(\'Downloading\',name,flush=True)\n            last_error=None\n            for attempt in range(1,4):\n                try:\n                    with urllib.request.urlopen(BASE_URL+name,timeout=45) as response:\n                        payload=response.read()\n                    break\n                except urllib.error.URLError as exc:\n                    last_error=exc\n                    print(f\'Network attempt {attempt}/3 failed: {exc}\',flush=True)\n                    if attempt < 3:time.sleep(2*attempt)\n            else:\n                raise RuntimeError(\n                    \'Could not download the pinned public author files. In Kaggle, \'\n                    \'open Settings, turn Internet on, then rerun this cell.\'\n                ) from last_error\n            if name in EXPECTED and hashlib.sha256(payload).hexdigest()!=EXPECTED[name]:\n                raise ValueError(f\'Inspected-source hash mismatch: {name}; stop and review this revision.\')\n            destination.write_bytes(payload)\n        digest=hashlib.sha256(destination.read_bytes()).hexdigest()\n        if name in EXPECTED and digest!=EXPECTED[name]:\n            raise ValueError(f\'Cached-source hash mismatch: {name}; use a fresh cache after review.\')\n        hashes[name]=digest\n    return hashes\n\n\ndef load_module(name,path):\n    spec=importlib.util.spec_from_file_location(name,path)\n    module=importlib.util.module_from_spec(spec)\n    sys.modules[name]=module\n    spec.loader.exec_module(module)\n    return module\n\n\ndef load_author_components(source_root,output):\n    import numpy as np\n    import torch\n    import torch.nn as nn\n    # Namespace isolation avoids importing the repository\'s unrelated experiments.\n    package=types.ModuleType(\'labelinspect_author_simplex\')\n    package.__path__=[str(source_root/\'utils/Simplex\')]\n    sys.modules[package.__name__]=package\n    noise=load_module(package.__name__+\'.noise\',source_root/\'utils/Simplex/noise.py\')\n\n    original=(source_root/\'src/models/UModels/UDHVT.py\').read_text(encoding=\'utf8\')\n    replacements={\n      \'from torchvision import models\':\'# Removed unused torchvision.models import.\',\n      \'from timm.data import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD, IMAGENET_INCEPTION_MEAN, IMAGENET_INCEPTION_STD\':\'# Removed unused timm image constants.\',\n      \'from timm.models.helpers import build_model_with_cfg, named_apply, adapt_input_conv\':\'from timm.models._manipulate import named_apply\',\n      \'from timm.models.layers import trunc_normal_, lecun_normal_, to_2tuple\':\'from timm.layers import trunc_normal_, lecun_normal_, to_2tuple\',\n      \'from timm.models.registry import register_model\':\'# Removed unused timm registry import.\',\n    }\n    for old,new in replacements.items():\n        if original.count(old)!=1:raise ValueError(\'Compatibility patch no longer matches the inspected source: \'+old)\n        original=original.replace(old,new)\n    patched=output/\'author_UDHVT_compat.py\'\n    patched.write_text(\'import numpy as np\\n\'+original,encoding=\'utf8\')\n    model_module=load_module(\'labelinspect_author_model\',patched)\n\n    # Keep the author definitions, including its variance convention. Avoid the\n    # top-level imports for unused image losses, plotting, datasets and backbones.\n    tree=ast.parse((source_root/\'GaussianDiffusion.py\').read_text(encoding=\'utf8\'))\n    wanted={\'get_beta_schedule\',\'extract\',\'mean_flat\',\'generate_simplex_4noise\',\'GaussianDiffusionModel\'}\n    selected=[node for node in tree.body if isinstance(node,(ast.FunctionDef,ast.ClassDef)) and node.name in wanted]\n    if {node.name for node in selected}!=wanted:raise ValueError(\'Required author diffusion definitions are missing\')\n    reduced=ast.Module(body=selected,type_ignores=[])\n    namespace={\'np\':np,\'torch\':torch,\'nn\':nn,\'OpenSimplex\':noise.OpenSimplex}\n    exec(compile(reduced,\'author_diffusion_l2_subset.py\',\'exec\'),namespace)\n    (output/\'author_diffusion_l2_subset.py\').write_text(ast.unparse(reduced),encoding=\'utf8\')\n\n    class NoisePredictionAdapter(nn.Module):\n        def __init__(self,backbone):super().__init__();self.backbone=backbone\n        def forward(self,x,t,y=None):\n            if y is not None:raise ValueError(\'This initial adapter supports the normal-only, unconditioned path\')\n            result=self.backbone(x,t,y=None)\n            prediction=result[0] if isinstance(result,tuple) else result\n            if prediction.shape!=x.shape:raise ValueError(\'Noise prediction does not match input shape\')\n            return prediction\n\n    return model_module,namespace,NoisePredictionAdapter,{\n      \'imports\':replacements,\'extra_import\':\'numpy for the author PositionalEmbedding helper\',\n      \'adapter\':\'Select tuple element 0; preserve the backbone computation.\',\n      \'diffusion_loading\':\'AST-load only the author definitions required for Gaussian/Tsimplex L2 and sampling; other losses are not supported.\',\n      \'sampling\':\'Pass denoise_fn=noise_fn so reverse steps use configured O/mu/p instead of the author alternate branch defaults.\',\n      \'calling_convention\':\'Set author diffusion train=False to select model(x,t,y=lab) during sampling. This is a dispatch flag; the model is explicitly switched with model.train()/eval().\',\n    }\n\n\ndef synthetic_batch(size,batch,device):\n    import numpy as np\n    import torch\n    from PIL import Image,ImageDraw,ImageFont\n    im=Image.new(\'L\',(size,size),235);draw=ImageDraw.Draw(im)\n    try:font=ImageFont.truetype(\'DejaVuSans.ttf\',20)\n    except OSError:font=ImageFont.load_default(size=20)\n    draw.rectangle((12,12,size-12,size-12),outline=20,width=2)\n    draw.text((24,45),\'LABEL A-104\',font=font,fill=20)\n    draw.text((24,90),\'BATCH 2026\',font=font,fill=20)\n    x=torch.from_numpy(np.asarray(im).copy()).float()/127.5-1\n    return x[None,None].repeat(batch,3,1,1).to(device)\n\n\ndef save_preview(x,reconstructed,destination):\n    from PIL import Image,ImageDraw\n    import numpy as np\n    images=[]\n    for tensor in [x,reconstructed]:\n        array=((tensor[0].detach().float().cpu().permute(1,2,0).numpy()+1)/2*255).clip(0,255).astype(np.uint8)\n        images.append(Image.fromarray(array))\n    sheet=Image.new(\'RGB\',(520,302),\'white\');draw=ImageDraw.Draw(sheet)\n    draw.text((12,10),\'INTEGRATION CHECK ONLY - TWO UPDATES\',fill=\'darkred\')\n    draw.text((12,32),\'Synthetic input\',fill=\'black\');draw.text((268,32),\'8-step reconstruction\',fill=\'black\')\n    for i,im in enumerate(images):sheet.paste(im.resize((224,224)),(12+i*256,54))\n    draw.text((12,283),\'No anomaly-removal or accuracy claim.\',fill=\'darkred\');sheet.save(destination)\n\n\ndef run(output,cache=None):\n    import importlib.metadata\n    import numpy as np\n    import torch\n    import numba\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    if not torch.cuda.is_available():raise RuntimeError(\'Select a GPU accelerator before running this notebook\')\n    cache=Path(cache) if cache else output/\'upstream\'/COMMIT\n    hashes=fetch_sources(cache)\n    random.seed(230224);np.random.seed(230224);torch.manual_seed(230224)\n    numba.set_num_threads(min(2,numba.get_num_threads()))\n    module,ns,adapter_type,patches=load_author_components(cache,output)\n    config={\'img_size\':224,\'patch_size\':16,\'in_chans\':3,\'embed_dim\':384,\'depth\':12,\n            \'num_heads\':6,\'mlp_ratio\':4.,\'num_classes\':None,\'mlp_time_embed\':True,\n            \'use_dec\':[\'DAFF\',\'DAFF\',\'DAFF\'],\'PE_type\':\'SPE\',\'refinement\':True,\'qkv_bias\':False}\n    report={\'status\':\'running\',\'scope\':\'integration_check_only\',\'upstream_commit\':COMMIT,\n            \'source_sha256\':hashes,\'compatibility_changes\':patches,\'model_configuration\':config,\n            \'configuration_note\':\'Illustrated SPE/DMHA/HFF/refinement variant. Code depth=12 gives six encoder blocks, one middle, six decoder blocks. This is not asserted to match every paper table.\',\n            \'torch\':torch.__version__,\'gpu\':torch.cuda.get_device_name(0),\n            \'gpu_vram_gib\':torch.cuda.get_device_properties(0).total_memory/2**30,\n            \'packages\':{n:importlib.metadata.version(n) for n in [\'timm\',\'einops\',\'numba\',\'numpy\']},\n            \'noise_parameters\':{\'octave\':6,\'frequency\':64,\'persistence\':.9},\n            \'training_steps\':2,\'batch_size\':2,\'total_diffusion_steps\':1000,\'reconstruction_steps\':8}\n    (output/\'integration_report.json\').write_text(json.dumps(report,indent=2))\n    device=torch.device(\'cuda:0\');torch.cuda.reset_peak_memory_stats()\n    print(\'Building author DTU-Net, width 384, six attention heads...\',flush=True)\n    backbone=module.UDHVT(**config).to(device);model=adapter_type(backbone)\n    report[\'parameter_count\']=sum(p.numel() for p in model.parameters())\n    x=synthetic_batch(224,2,device)\n    t=torch.tensor([50,150],device=device,dtype=torch.long)\n    diffusion=ns[\'GaussianDiffusionModel\']([224,224],ns[\'get_beta_schedule\'](1000,\'cosine\'),img_channels=3,\n                 loss_type=\'l2\',noise=\'4dsimplex\',octave=6,frequency=64,persistence=.9,train=False)\n    print(\'Compiling the author 4D noise function on CPU; first use may take a few minutes...\',flush=True)\n    start=time.perf_counter()\n    probe=torch.zeros(1,1,4,4,device=device)\n    diffusion.noise_fn(probe,torch.tensor([5],device=device))\n    report[\'noise_first_compile_seconds\']=time.perf_counter()-start\n    noise=diffusion.noise_fn(x,t).float()\n    assert noise.shape==x.shape and torch.isfinite(noise).all()\n    assert not torch.allclose(noise[0],noise[1]),\'Different time coordinates unexpectedly generated identical samples\'\n    report[\'noise_shape\']=list(noise.shape)\n    report[\'noise_mean\']=float(noise.mean());report[\'noise_std\']=float(noise.std())\n    report[\'noise_normalization\']=\'Author raw amplitude retained; no per-sample standardization.\'\n    report[\'batch_noise_note\']=\'The author generator uses t as the fourth coordinate. Duplicate time coordinates with one seed can produce identical noise across batch entries; this remains to be assessed during training.\'\n    model.train();optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=0.)\n    report[\'losses\']=[];report[\'gradient_norms\']=[]\n    tracked=backbone.pos_embed.detach().clone()\n    for step in range(2):\n        optimizer.zero_grad(set_to_none=True)\n        losses,noisy,predicted=diffusion.calc_loss(model,x,None,t)\n        loss=losses[\'loss\'].mean()\n        assert predicted.shape==x.shape and torch.isfinite(loss)\n        loss.backward()\n        grads=[p.grad for p in model.parameters() if p.grad is not None]\n        assert grads and all(torch.isfinite(g).all() for g in grads)\n        norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n        optimizer.step()\n        report[\'losses\'].append(float(loss.detach()))\n        report[\'gradient_norms\'].append(float(norm))\n        print(f\'Update {step+1}/2 passed; L2 noise loss {float(loss.detach()):.6f}\',flush=True)\n    assert not torch.equal(tracked,backbone.pos_embed.detach()),\'Optimizer did not change the tracked parameter\'\n    report[\'tracked_parameter_changed\']=True\n    report[\'parameters_without_grad\']=[name for name,p in model.named_parameters() if p.grad is None]\n    model.eval()\n    print(\'Checking eight author reverse-diffusion steps. This model is not trained for detection.\',flush=True)\n    with torch.no_grad():\n        result=diffusion.forward_backward(model,x[:1],None,see_whole_sequence=None,t_distance=8,denoise_fn=\'noise_fn\')\n    assert result.shape==x[:1].shape and torch.isfinite(result).all()\n    residual=(x[:1]-result).square().mean(dim=1)\n    assert residual.shape==(1,224,224) and torch.isfinite(residual).all()\n    save_preview(x,result,output/\'integration_preview.png\')\n    report[\'reconstruction_shape\']=list(result.shape);report[\'residual_shape\']=list(residual.shape)\n    report[\'peak_gpu_allocated_gib\']=torch.cuda.max_memory_allocated()/2**30\n    report[\'peak_gpu_reserved_gib\']=torch.cuda.max_memory_reserved()/2**30\n    report[\'status\']=\'passed\'\n    report[\'not_completed\']=[\'Training a useful anomaly model\',\'Real label dataset\',\'Paper metrics reproduction\',\'Quality comparison with CPU baseline\']\n    (output/\'integration_report.json\').write_text(json.dumps(report,indent=2))\n    print(\'\\nDTU-NET + TSIMPLEX INTEGRATION CHECK PASSED\',flush=True)\n    print(json.dumps({k:report[k] for k in [\'parameter_count\',\'noise_shape\',\'losses\',\'reconstruction_shape\',\'peak_gpu_allocated_gib\',\'peak_gpu_reserved_gib\']},indent=2))\n    print(\'Saved:\',output/\'integration_report.json\',flush=True)\n    return report\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--output\',default=\'artifacts/author_integration\')\n    parser.add_argument(\'--cache\',default=None)\n    args=parser.parse_args();run(args.output,args.cache)\n', encoding='utf8')
print('Wrote:', script)


In [ ]:
spec=importlib.util.spec_from_file_location('labelinspect_author_smoke',WORK/'author_smoke.py')
author=importlib.util.module_from_spec(spec);sys.modules[spec.name]=author;spec.loader.exec_module(author)
import numba
numba.set_num_threads(min(2,numba.get_num_threads()))
source_cache=WORK/'upstream'/author.COMMIT
hashes=author.fetch_sources(source_cache)
model_module,diffusion_ns,Adapter,compatibility=author.load_author_components(source_cache,OUTPUT)
print('Pinned author commit:',author.COMMIT)


In [ ]:
IMAGE_EXTENSIONS={'.jpg','.jpeg','.png'}
val_paths=sorted(path for path in (DATA_ROOT/'validation'/'normal').iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)
assert len(val_paths)==10
assert {path.stem.split('_')[0] for path in val_paths}=={'N09','N10'}
RESAMPLE=getattr(Image,'Resampling',Image).BILINEAR
NEAREST=getattr(Image,'Resampling',Image).NEAREST

def pad_image(image,resample=RESAMPLE,fill=(238,238,238)):
    fitted=ImageOps.contain(image,(IMAGE_SIZE,IMAGE_SIZE),resample)
    canvas=Image.new(image.mode,(IMAGE_SIZE,IMAGE_SIZE),fill)
    canvas.paste(fitted,((IMAGE_SIZE-fitted.width)//2,(IMAGE_SIZE-fitted.height)//2))
    return canvas

def tensor_from_image(image):
    array=np.asarray(pad_image(image.convert('RGB')),dtype=np.float32).copy()/127.5-1.
    return torch.from_numpy(array).permute(2,0,1)

def synthetic_defect(source,kind,index):
    image=source.convert('RGB')
    changed=image.copy()
    width,height=changed.size
    if kind=='missing_print':
        # Remove a different narrow barcode interval for each source image.
        x0=int(width*(0.25+0.035*(index%8))); x1=x0+int(width*0.055)
        y0=int(height*0.60); y1=int(height*0.82)
        ImageDraw.Draw(changed).rectangle((x0,y0,x1,y1),fill=(239,239,239))
    elif kind=='smudge':
        overlay=Image.new('RGBA',changed.size,(0,0,0,0))
        draw=ImageDraw.Draw(overlay)
        x0=int(width*(0.52+0.02*(index%4))); y0=int(height*(0.28+0.025*(index%3)))
        draw.ellipse((x0,y0,x0+int(width*.12),y0+int(height*.12)),fill=(35,35,35,165))
        changed=Image.alpha_composite(changed.convert('RGBA'),overlay).convert('RGB')
    elif kind=='tear':
        y=int(height*(0.34+0.035*(index%6)))
        ImageDraw.Draw(changed).polygon([
            (width-1,y-int(height*.10)),(width-1,y+int(height*.13)),
            (int(width*.86),y+int(height*.035)),(int(width*.92),y-int(height*.03)),
        ],fill=(205,205,205))
    else: raise ValueError(kind)
    before=np.asarray(image,dtype=np.int16)
    after=np.asarray(changed,dtype=np.int16)
    mask=Image.fromarray((np.max(np.abs(after-before),axis=2)>8).astype(np.uint8)*255)
    mask_array=np.asarray(pad_image(mask,resample=NEAREST,fill=0))>0
    return tensor_from_image(changed),mask_array

normal_tensors=[]
synthetic_tensors=[]
synthetic_masks=[]
synthetic_kinds=[]
synthetic_names=[]
for index,path in enumerate(val_paths):
    with Image.open(path) as opened: source=opened.convert('RGB')
    normal_tensors.append(tensor_from_image(source))
    for kind in ['missing_print','smudge','tear']:
        tensor,mask=synthetic_defect(source,kind,index)
        assert mask.any(),(path,kind)
        synthetic_tensors.append(tensor);synthetic_masks.append(mask)
        synthetic_kinds.append(kind);synthetic_names.append(path.stem+'__'+kind)
normal_tensors=torch.stack(normal_tensors)
synthetic_tensors=torch.stack(synthetic_tensors)
synthetic_masks=np.stack(synthetic_masks)

label_roi=np.zeros((IMAGE_SIZE,IMAGE_SIZE),dtype=bool)
resized_height=round(650*IMAGE_SIZE/1063);roi_top=(IMAGE_SIZE-resized_height)//2
label_roi[roi_top:roi_top+resized_height,:]=True
assert normal_tensors.shape==(10,3,224,224) and synthetic_tensors.shape==(30,3,224,224)
print('Prepared',len(normal_tensors),'real normals and',len(synthetic_tensors),'synthetic validation defects.')


In [ ]:
device=torch.device('cuda:0')
checkpoint=torch.load(CHECKPOINT,map_location=device,weights_only=False)
assert checkpoint['step']==2000
assert checkpoint['author_commit']==author.COMMIT
assert checkpoint.get('dataset_category')=='printed_label_train_v1'
model_config=checkpoint['model_config']
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
backbone=model_module.UDHVT(**model_config).to(device);model=Adapter(backbone)
model.load_state_dict(checkpoint['model']);model.eval()
diffusion=diffusion_ns['GaussianDiffusionModel'](
    [224,224],diffusion_ns['get_beta_schedule'](1000,'cosine'),img_channels=3,
    loss_type='l2',noise='4dsimplex',octave=6,frequency=64,persistence=.9,train=False)
diffusion.noise_fn(torch.zeros(1,1,4,4,device=device),torch.tensor([5],device=device))
torch.cuda.reset_peak_memory_stats()
print('Loaded frozen step',checkpoint['step'],'checkpoint.')


In [ ]:
def reconstruct(tensors,distance):
    outputs=[];elapsed=0.
    for start in range(0,len(tensors),BATCH_SIZE):
        x=tensors[start:start+BATCH_SIZE].to(device)
        tick=time.perf_counter()
        with torch.inference_mode():
            result=diffusion.forward_backward(model,x,None,see_whole_sequence=None,
                                               t_distance=distance,denoise_fn='noise_fn')
        torch.cuda.synchronize();elapsed+=time.perf_counter()-tick
        outputs.append(result.cpu())
    return torch.cat(outputs),elapsed

def residual(inputs,recons):
    return (inputs-recons).square().mean(dim=1).numpy().astype(np.float32)

def mask_metrics(pred,truth):
    pred=np.asarray(pred,dtype=bool);truth=np.asarray(truth,dtype=bool)
    tp=int(np.sum(pred&truth));fp=int(np.sum(pred&~truth));fn=int(np.sum(~pred&truth))
    return {
        'dice':2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 1.,
        'iou':tp/(tp+fp+fn) if tp+fp+fn else 1.,
    }

all_results=[];detail_rows=[];cached={}
for distance in DISTANCES:
    # Reset seeds before each candidate so this comparison is reproducible.
    random.seed(SEED+distance);np.random.seed(SEED+distance);torch.manual_seed(SEED+distance);torch.cuda.manual_seed_all(SEED+distance)
    normal_recon,normal_seconds=reconstruct(normal_tensors,distance)
    defect_recon,defect_seconds=reconstruct(synthetic_tensors,distance)
    normal_maps=residual(normal_tensors,normal_recon)
    defect_maps=residual(synthetic_tensors,defect_recon)
    pixel_threshold=float(np.quantile(normal_maps[:,label_roi],1-PIXEL_NORMAL_FPR,method='higher'))
    normal_image_scores=np.quantile(normal_maps[:,label_roi],IMAGE_SCORE_QUANTILE,axis=1)
    image_threshold=float(np.max(normal_image_scores))
    defect_image_scores=np.quantile(defect_maps[:,label_roi],IMAGE_SCORE_QUANTILE,axis=1)
    predicted=(defect_maps>pixel_threshold)&label_roi[None]
    per_defect=[]
    for index,(name,kind) in enumerate(zip(synthetic_names,synthetic_kinds)):
        metrics=mask_metrics(predicted[index],synthetic_masks[index])
        detected=bool(defect_image_scores[index]>image_threshold)
        row={'distance':distance,'file':name,'kind':kind,'image_score':float(defect_image_scores[index]),
             'detected':detected,**metrics}
        detail_rows.append(row);per_defect.append(row)
    result={
        'distance':distance,
        'pixel_threshold':pixel_threshold,
        'image_threshold':image_threshold,
        'normal_pixel_fpr':float(((normal_maps>pixel_threshold)&label_roi[None])[:,label_roi].mean()),
        'normal_image_fpr':float(np.mean(normal_image_scores>image_threshold)),
        'synthetic_mean_dice':float(np.mean([row['dice'] for row in per_defect])),
        'synthetic_mean_iou':float(np.mean([row['iou'] for row in per_defect])),
        'synthetic_image_sensitivity':float(np.mean([row['detected'] for row in per_defect])),
        'mean_seconds_per_image':float((normal_seconds+defect_seconds)/(len(normal_tensors)+len(synthetic_tensors))),
        'by_kind':{},
    }
    for kind in ['missing_print','smudge','tear']:
        subset=[row for row in per_defect if row['kind']==kind]
        result['by_kind'][kind]={
            'mean_dice':float(np.mean([row['dice'] for row in subset])),
            'image_sensitivity':float(np.mean([row['detected'] for row in subset])),
        }
    all_results.append(result)
    cached[distance]=(normal_recon,defect_recon,normal_maps,defect_maps,predicted)
    print(json.dumps(result,indent=2))

selected=max(all_results,key=lambda item:(item['synthetic_mean_dice'],item['synthetic_image_sensitivity'],-item['distance']))
selection={
    'status':'passed',
    'scope':'validation_only_model_selection',
    'real_test_images_used':0,
    'checkpoint_step':int(checkpoint['step']),
    'author_commit':author.COMMIT,
    'candidate_distances':DISTANCES,
    'selection_rule':'highest mean synthetic-defect Dice; then sensitivity; then shorter distance',
    'selected':selected,
    'all_candidates':all_results,
    'synthetic_validation_warning':'Digitally generated validation defects support parameter selection but do not estimate real-defect accuracy.',
    'frozen_for_real_test':['checkpoint','registration','input transform','selected distance','pixel threshold','image score','image threshold'],
}
(OUTPUT/'model_selection.json').write_text(json.dumps(selection,indent=2))
with (OUTPUT/'synthetic_validation_per_image.csv').open('w',newline='') as stream:
    writer=csv.DictWriter(stream,fieldnames=list(detail_rows[0]));writer.writeheader();writer.writerows(detail_rows)
print('SELECTED CONFIGURATION')
print(json.dumps(selected,indent=2))


In [ ]:
distance=selected['distance']
normal_recon,defect_recon,normal_maps,defect_maps,predicted=cached[distance]
def show_tensor(tensor): return ((tensor.permute(1,2,0).numpy()+1)/2).clip(0,1)

indices=[0,1,2]
fig,axes=plt.subplots(3,5,figsize=(15,9))
for row,index in enumerate(indices):
    axes[row,0].imshow(show_tensor(synthetic_tensors[index]));axes[row,0].set_title(synthetic_kinds[index])
    axes[row,1].imshow(show_tensor(defect_recon[index]));axes[row,1].set_title('Reconstruction')
    axes[row,2].imshow(defect_maps[index],cmap='magma');axes[row,2].set_title('Residual')
    axes[row,3].imshow(predicted[index],cmap='gray',vmin=0,vmax=1);axes[row,3].set_title('Prediction')
    axes[row,4].imshow(synthetic_masks[index],cmap='gray',vmin=0,vmax=1);axes[row,4].set_title('Known synthetic mask')
    for axis in axes[row]: axis.axis('off')
fig.suptitle(f'Validation-only distance selection — selected t={distance}')
fig.tight_layout();fig.savefig(OUTPUT/'model_selection_preview.png',dpi=160,bbox_inches='tight');plt.show()

archive=shutil.make_archive('/kaggle/working/printed_label_model_selection_results','zip',OUTPUT)
print('\nMODEL SELECTION PASSED')
print('Download:',archive)
print('Do not attach the real test dataset until this output is saved.')
